# KAVALAN — OCR Fine-tune (TrOCR on handwritten/scanned text)

**Run this as a Kaggle Notebook**, not locally — see `ml/README.md` in the repo for setup steps.

1. Add Input dataset: `appenlimited/handwritten-ocr-image-data-in-english`
2. Settings → Accelerator → GPU T4 x2
3. Run All

Goal: fine-tune `microsoft/trocr-base-handwritten` so it transcribes autopsy report
images/scans at least as well as the current Groq vision call in
`src/app/api/analyze/autopsy/extract-image/route.ts`, then export to ONNX so it can run
locally via `onnxruntime-node` instead of an external API call.

In [ ]:
!pip install -q transformers datasets jiwer optimum[exporters] pillow accelerate

## Step 1 — Inspect what Kaggle actually mounted

Kaggle dataset internal layouts vary by uploader even for nominally similar datasets.
**Run this cell first** and read the printed tree before touching Step 2 — adjust the
paths/column names in Step 2 to match what you actually see here.

In [ ]:
import os

INPUT_ROOT = "/kaggle/input"
for root, dirs, files in os.walk(INPUT_ROOT):
    depth = root.replace(INPUT_ROOT, "").count(os.sep)
    if depth > 3:
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/")
    if depth == 3:
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... ({len(files)} files total)")

## Step 2 — Build the (image_path, text) pairs list

This dataset is typically an images folder plus a labels file (CSV/JSON/TSV) mapping
filename → transcribed text. **Adjust `DATASET_DIR`, `LABELS_FILE`, and the column
names below** to match what Step 1 printed.

In [ ]:
import glob
import pandas as pd

# TODO: adjust to match Step 1's output.
DATASET_DIR = "/kaggle/input/handwritten-ocr-image-data-in-english"

# Try to auto-locate a labels file (csv/tsv/json) — override LABELS_FILE manually if
# this guesses wrong.
candidates = (
    glob.glob(f"{DATASET_DIR}/**/*.csv", recursive=True)
    + glob.glob(f"{DATASET_DIR}/**/*.tsv", recursive=True)
    + glob.glob(f"{DATASET_DIR}/**/*.json", recursive=True)
)
print("Candidate label files found:", candidates)
LABELS_FILE = candidates[0] if candidates else None

if LABELS_FILE and LABELS_FILE.endswith(".json"):
    df = pd.read_json(LABELS_FILE)
elif LABELS_FILE:
    sep = "\t" if LABELS_FILE.endswith(".tsv") else ","
    df = pd.read_csv(LABELS_FILE, sep=sep)
else:
    raise FileNotFoundError(
        "No labels file auto-detected. Set LABELS_FILE manually after inspecting Step 1's output."
    )

print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
# TODO: rename these to the actual column names printed above.
IMAGE_COLUMN = "filename"
TEXT_COLUMN = "text"

# Resolve image column to full paths — try a few common locations.
image_dirs = [d for d in glob.glob(f"{DATASET_DIR}/**/", recursive=True)]


def resolve_image_path(name: str) -> str | None:
    for d in image_dirs:
        candidate = os.path.join(d, name)
        if os.path.isfile(candidate):
            return candidate
    return None


df["resolved_path"] = df[IMAGE_COLUMN].apply(resolve_image_path)
unresolved = df["resolved_path"].isna().sum()
print(f"Resolved {len(df) - unresolved}/{len(df)} image paths")
df = df.dropna(subset=["resolved_path"]).reset_index(drop=True)

from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"train={len(train_df)} eval={len(eval_df)}")

## Step 3 — Dataset + processor

In [ ]:
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor

MODEL_NAME = "microsoft/trocr-base-handwritten"
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)


class OCRDataset(Dataset):
    def __init__(self, df, processor, max_target_length=128):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["resolved_path"]).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            str(row[TEXT_COLUMN]),
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
        ).input_ids
        labels = [
            l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels
        ]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}


train_dataset = OCRDataset(train_df, processor)
eval_dataset = OCRDataset(eval_df, processor)
print(len(train_dataset), len(eval_dataset))

## Step 4 — Model + training

`EPOCHS` is deliberately small so a first run fits inside a Kaggle session and proves
the pipeline works end to end. Bump it up (e.g. 10–20) for a real fine-tune once this
completes cleanly.

In [ ]:
from transformers import VisionEncoderDecoderModel

model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 128
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

In [ ]:
import evaluate

cer_metric = evaluate.load("cer")


def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

EPOCHS = 3  # bump up once the pipeline is confirmed working end to end

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/trocr-checkpoints",
    predict_with_generate=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=torch.cuda.is_available(),
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.feature_extractor,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()

## Step 5 — Sanity-check a few predictions

In [ ]:
import random

sample_idx = random.sample(range(len(eval_df)), min(5, len(eval_df)))
for i in sample_idx:
    row = eval_df.iloc[i]
    image = Image.open(row["resolved_path"]).convert("RGB")
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(model.device)
    generated_ids = model.generate(pixel_values)
    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print("GT:  ", row[TEXT_COLUMN])
    print("PRED:", pred_text)
    print("---")

## Step 6 — Save + export to ONNX

Uses `optimum` (Hugging Face's export tooling) since TrOCR is an encoder-decoder model
and needs the encoder/decoder graphs exported together with the merged decoder for
generation to work correctly in `onnxruntime`.

In [ ]:
FINETUNED_DIR = "/kaggle/working/trocr-finetuned"
model.save_pretrained(FINETUNED_DIR)
processor.save_pretrained(FINETUNED_DIR)

ONNX_DIR = "/kaggle/working/trocr-onnx"
!optimum-cli export onnx --model {FINETUNED_DIR} --task image-to-text-with-past {ONNX_DIR}

print("Done. Download the /kaggle/working/trocr-onnx directory from the Output tab.")